## Network Intrusion Project

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
# 1. Load dataset
data = pd.read_csv("networkintrusion.csv")

In [6]:
# 2. Check basic info
print(data.head())

   duration protocol_type   service  flag  src_bytes  dst_bytes  land  \
0         0           tcp   private   REJ        0.0        0.0     0   
1         0           tcp   private   REJ        0.0        0.0     0   
2         2           tcp  ftp_data    SF    12983.0        0.0     0   
3         0          icmp     eco_i    SF       20.0        0.0     0   
4         1           tcp    telnet  RSTO        0.0       15.0     0   

   wrong_fragment  urgent  hot  ...  dst_host_srv_count  \
0               0       0    0  ...                  10   
1               0       0    0  ...                   1   
2               0       0    0  ...                  86   
3               0       0    0  ...                  57   
4               0       0    0  ...                  86   

   dst_host_same_srv_rate  dst_host_diff_srv_rate  \
0                    0.04                    0.06   
1                    0.00                    0.06   
2                    0.61                    0.

In [7]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   duration                     22544 non-null  int64  
 1   protocol_type                22544 non-null  object 
 2   service                      22544 non-null  object 
 3   flag                         22544 non-null  object 
 4   src_bytes                    22538 non-null  float64
 5   dst_bytes                    22538 non-null  float64
 6   land                         22544 non-null  int64  
 7   wrong_fragment               22544 non-null  int64  
 8   urgent                       22544 non-null  int64  
 9   hot                          22544 non-null  int64  
 10  num_failed_logins            22544 non-null  int64  
 11  logged_in                    22544 non-null  int64  
 12  num_compromised              22544 non-null  int64  
 13  root_shell      

In [8]:
print(data["class"].value_counts())

class
anomaly    12833
normal      9711
Name: count, dtype: int64


In [9]:
# 3. Separate features and target
X = data.drop("class", axis=1)
y = data["class"]

In [10]:
# Optional: map target labels
y = y.map({
    "normal": 0,
    "anomaly": 1
})

In [11]:
# 4. Detect categorical and numeric columns
categorical_cols = ["protocol_type", "service", "flag"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

In [12]:
# 5. Preprocessing for numeric columns
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [13]:
# 6. Preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [14]:
# 7. Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [15]:
# 8. Build model pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [16]:
# 9. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [17]:
# 10. Train model
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['duration', 'src_bytes',
                                                   'dst_bytes', 'land',
                                                   'wrong_fragment', 'urgent',
                                                   'hot', 'num_failed_logins',
                                                   'logged_in',
                                                   'num_compromised',
                                                   'root_shell', 'num_root',
                                                   'num_shells',
                                                   'num_access_files',
                                                   'num_ou...
                                                   'same_srv_rate',
                                                   'diff_srv_rate',
                                                   'srv_diff_host_rate',
                                                   'dst_host_count',
                                                   'dst_host_srv_count',
                                                   'dst_host_same_srv_rate',
                                                   'dst_host_diff_srv_rate', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['protocol_type', 'service',
                                                   'flag'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [18]:
# 11. Predict
y_pred = model.predict(X_test)

In [19]:
# 12. Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9498780217343091


In [20]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[1815  127]
 [  99 2468]]


In [21]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["normal", "anomaly"]))


Classification Report:
              precision    recall  f1-score   support

      normal       0.95      0.93      0.94      1942
     anomaly       0.95      0.96      0.96      2567

    accuracy                           0.95      4509
   macro avg       0.95      0.95      0.95      4509
weighted avg       0.95      0.95      0.95      4509

